In [17]:
import pandas as pd
from pathlib import Path
import re

ROOT = Path("/home/py/groundwater/data/acis")

In [12]:
ACIS_RAW = ROOT / "raw"
ACIS_CLEAN = ROOT / "processed"

In [13]:
files = sorted(ACIS_RAW.glob("ACISDailyData-*.csv"))
print(f"{len(files)} files")

8 files


In [14]:
acis = pd.concat((pd.read_csv(f, dtype=str) for f in files), ignore_index=True)
acis["date"]   = pd.to_datetime(acis["Date (Local Standard Time)"], format="%d-%B-%Y")
acis["precip"] = pd.to_numeric(acis["Precip. (mm)"], errors="coerce")

In [15]:
acis = (acis.sort_values("Precip. Source Flag", na_position="last")
             .drop_duplicates(subset=["Station Name", "date"], keep="first"))

In [18]:
summary = []
for name, g in acis.groupby("Station Name"):
    g = g.sort_values("date").set_index("date")

    # trim leading/trailing stretches where the station wasn't reporting at all
    served = g["Precip. Source Flag"].notna()
    if not served.any():
        print(f"skip {name}: no observations")
        continue
    g = g.loc[served.idxmax(): served[::-1].idxmax()]

    # continuous daily axis
    g = g.reindex(pd.date_range(g.index.min(), g.index.max(), freq="D"))
    g.index.name = "date"

    out = g[["precip", "Precip. Source Flag", "Precip. Comment"]].rename(
        columns={"Precip. Source Flag": "flag", "Precip. Comment": "source"})
    out.insert(0, "station", name)

    slug = re.sub(r"\W+", "_", name).strip("_")
    out.to_csv(ACIS_CLEAN / f"{slug}.csv")

    summary.append({"station": name, "start": out.index.min(), "end": out.index.max(),
                    "n_days": len(out), "pct_missing": out["precip"].isna().mean().round(3)})

pd.DataFrame(summary)

,station,start,end,n_days,pct_missing
0,Bow Valley,2005-04-01,2026-08-13,7805,0.001
1,Canmore Auto,2024-07-15,2026-08-13,760,0.495
2,Canmore Civic Centre,2023-12-09,2026-08-13,979,0.053
3,Kananaskis Boundary Auto,2009-07-26,2026-08-13,6228,0.547
